# Decentralized Value Learning - Debug Step 1

**Step 1: Nobody picks apples**
- Agents teleport to NON-apple locations only
- All rewards = 0
- True value = 0 for all states

Goal: Verify TD0 converges to 0 with 5-channel encoding.

In [ ]:
# =============================================================================
# PARAMETERS
# =============================================================================

# Environment
HEIGHT = 9
WIDTH = 9
NUM_AGENTS = 4
NUM_APPLES = 40
PICKER_REWARD = -1

# Training
LR = 0.001
GAMMA = 0.99
TRAIN_STEPS = 1000
EVAL_FREQ = 50

# Model
HIDDEN_DIM = 64
NUM_LAYERS = 2

# Evaluation
NUM_TEST_CASES = 200

# Output
OUTPUT_DIR = "outputs"
SEED = 10203 # set to -1 to randomize

In [2]:
# =============================================================================
# IMPORTS & SETUP
# =============================================================================

import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import psutil
import os
from dataclasses import dataclass
from typing import Dict, List, Tuple, Callable, Optional
from pathlib import Path
import json 

from tadd_helpers.setting_seed import set_all_seeds

# Seed everything
if SEED != -1:
    set_all_seeds(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Grid: {HEIGHT}x{WIDTH}, Agents: {NUM_AGENTS}, Apples: {NUM_APPLES}")

Device: cpu
Grid: 9x9, Agents: 4, Apples: 40


In [3]:
# =============================================================================
# STATE CLASS
# =============================================================================

@dataclass
class State:
    """MDP State: agent positions, current actor, apple grid, dimensions."""
    _agents: Dict[int, Tuple[int, int]]
    actor: int
    apples: np.ndarray
    H: int
    L: int
    
    def agent_position(self, agent_idx: int) -> Tuple[int, int]:
        return self._agents[agent_idx]
    
    def set_agent_position(self, agent_idx: int, pos: Tuple[int, int]) -> None:
        self._agents[agent_idx] = pos
    
    def copy(self) -> 'State':
        return State(
            _agents=self._agents.copy(),
            actor=self.actor,
            apples=self.apples.copy(),
            H=self.H,
            L=self.L
        )
    
    def __str__(self) -> str:
        lines = [f"State (H={self.H}, L={self.L}, actor={self.actor})"]
        lines.append("Agent positions:")
        for idx, pos in sorted(self._agents.items()):
            on_apple = self.apples[pos[0], pos[1]] > 0
            marker = " [ON APPLE]" if on_apple else ""
            actor_marker = " <-- ACTOR" if idx == self.actor else ""
            lines.append(f"  Agent {idx}: {pos}{marker}{actor_marker}")
        lines.append(f"Total apples: {int(self.apples.sum())}")
        return "\n".join(lines)

In [4]:
# =============================================================================
# ENVIRONMENT FUNCTIONS
# =============================================================================


def init_apple_grid(height: int, width: int, num_apples: int) -> np.ndarray:
    """Create fixed apple grid."""
    grid = np.zeros((height, width), dtype=np.float32)
    indices = np.random.choice(height * width, size=num_apples, replace=False)
    for idx in indices:
        r, c = divmod(idx, width)
        grid[r, c] = 1.0
    return grid


def get_valid_positions(apples: np.ndarray, on_apple: bool) -> List[Tuple[int, int]]:
    """Get positions where agents can teleport to. So for step 1, only positions with no apples (on_apple=False)."""
    if on_apple:
        return list(zip(*np.where(apples > 0)))
    else:
        return list(zip(*np.where(apples == 0)))


def init_state(height: int, width: int, num_agents: int, apples: np.ndarray, 
               valid_positions: Optional[List[Tuple[int, int]]]) -> State:
    """Initialize state with random agent positions and random actor."""
    agents = {}
    for i in range(num_agents):
        if valid_positions:
            pos = valid_positions[np.random.randint(len(valid_positions))]
            agents[i] = pos
        else:
            r = np.random.randint(0, height)
            c = np.random.randint(0, width)
            agents[i] = (r, c)
    actor = np.random.randint(0, num_agents)
    return State(_agents=agents, actor=actor, apples=apples.copy(), H=height, L=width)


def teleport_and_advance(state: State, valid_positions: List[Tuple[int, int]], num_agents: int) -> None:
    """Teleport current actor to random valid position, then pick next actor. Modifies in-place."""
    # Teleport current actor
    pos = valid_positions[np.random.randint(len(valid_positions))]
    state.set_agent_position(state.actor, pos)
    # Advance to next actor
    state.actor = np.random.randint(0, num_agents)


def get_reward(state: State) -> Dict[int, float]:
    """Decentralized reward: picker gets PICKER_REWARD, others split the rest."""
    num_agents = len(state._agents)
    actor_pos = state.agent_position(state.actor)
    hit_apple = state.apples[actor_pos[0], actor_pos[1]] > 0
    
    rewards = {i: 0.0 for i in range(num_agents)}
    if hit_apple:
        rewards[state.actor] = PICKER_REWARD
        if num_agents > 1:
            other_reward = (1.0 - PICKER_REWARD) / (num_agents - 1)
            for i in range(num_agents):
                if i != state.actor:
                    rewards[i] = other_reward
    return rewards

In [ ]:
# =============================================================================
# MODEL CLASS
# =============================================================================

class ValueMLP:
    """Simple MLP for value prediction. Pure online TD0 with MSE loss."""
    
    def __init__(
        self,
        self_agent_idx: int,
        height: int,
        width: int,
        hidden_dim: int,
        num_layers: int,
        lr: float,
        gamma: float,
        device: torch.device
    ):
        self.self_agent_idx = self_agent_idx
        self.H = height
        self.W = width
        self.gamma = gamma
        self.device = device
        
        # 5 channels: apples, others, self, self_is_actor, other_is_actor
        input_dim = 5 * height * width
        
        # Build network
        layers = []
        in_d = input_dim
        for _ in range(num_layers):
            layers.append(nn.Linear(in_d, hidden_dim))
            layers.append(nn.ReLU())
            in_d = hidden_dim
        layers.append(nn.Linear(in_d, 1))
        
        self.net = nn.Sequential(*layers).to(device)
        self.optimizer = torch.optim.Adam(self.net.parameters(), lr=lr)
    
    def encode(self, state: State) -> np.ndarray:
        """5-channel encoding: apples, others, self, self_is_actor, other_is_actor."""
        H, W = self.H, self.W
        i = self.self_agent_idx
        
        # Channel 0: apples
        c_apples = state.apples.astype(np.float32)
        
        # Channel 1: other agents
        c_others = np.zeros((H, W), dtype=np.float32)
        for agent_idx, pos in state._agents.items():
            if agent_idx != i:
                c_others[pos[0], pos[1]] += 1.0
        
        # Channel 2: self position
        c_self = np.zeros((H, W), dtype=np.float32)
        self_pos = state.agent_position(i)
        c_self[self_pos[0], self_pos[1]] = 1.0
        
        # Channel 3: self is actor
        c_self_act = np.zeros((H, W), dtype=np.float32)
        if state.actor == i:
            c_self_act[self_pos[0], self_pos[1]] = 1.0
        
        # Channel 4: other is actor
        c_other_act = np.zeros((H, W), dtype=np.float32)
        if state.actor != i:
            actor_pos = state.agent_position(state.actor)
            c_other_act[actor_pos[0], actor_pos[1]] = 1.0
        
        return np.stack([c_apples, c_others, c_self, c_self_act, c_other_act]).flatten()
    
    def get_value(self, state: State) -> float:
        """Get V(s) for this agent."""
        enc = self.encode(state)
        x = torch.tensor(enc, dtype=torch.float32, device=self.device).unsqueeze(0)
        with torch.no_grad():
            return self.net(x).item()
    
    def update(self, state: State, reward: float, next_state: State) -> Dict:
        """Pure online TD0 update. Returns TD error. Returns useful logging info."""
        # Encode
        s_enc = self.encode(state)
        ns_enc = self.encode(next_state)
        
        s_t = torch.tensor(s_enc, dtype=torch.float32, device=self.device).unsqueeze(0)
        ns_t = torch.tensor(ns_enc, dtype=torch.float32, device=self.device).unsqueeze(0)
        r_t = torch.tensor([[reward]], dtype=torch.float32, device=self.device)
        
        # Compute TD target
        with torch.no_grad():
            v_next = self.net(ns_t)
            td_target = r_t + self.gamma * v_next
        
        # Compute current value and loss
        v_curr = self.net(s_t)
        loss = nn.MSELoss()(v_curr, td_target)
        
        # Optimize
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.net.parameters(), 1.0)
        self.optimizer.step()
        
        return {
            'loss': loss.item(),
            'v_curr': v_curr.item(),
            'td_target': td_target.item(),
        }

In [6]:
# =============================================================================
# TEST CASES
# =============================================================================

@dataclass
class TestCase:
    """A test case: state + true values per agent."""
    state: State
    true_values: Dict[int, float]


def generate_test_cases(
    n: int,
    height: int,
    width: int,
    num_agents: int,
    apples: np.ndarray,
    true_value_fn: Callable[[State], Dict[int, float]],
    valid_positions: Optional[List[Tuple[int, int]]]
) -> List[TestCase]:
    """Generate n random test cases."""
    cases = []
    for _ in range(n):
        state = init_state(height, width, num_agents, apples, valid_positions)
        true_values = true_value_fn(state)
        cases.append(TestCase(state=state, true_values=true_values))
    return cases


# Step 1: All true values are 0
def true_value_fn_step1(state: State) -> Dict[int, float]:
    return {i: 0.0 for i in state._agents.keys()}

In [7]:
# =============================================================================
# EVALUATION
# =============================================================================

def evaluate(models: List[ValueMLP], test_cases: List[TestCase]) -> Dict:
    """
    Evaluate models on test cases.
    Returns dict with per-agent and overall metrics.
    """
    num_agents = len(models)
    
    EPS = 1e-9
    
    # Collect predictions and true values per agent
    agent_preds = {i: [] for i in range(num_agents)}
    agent_trues = {i: [] for i in range(num_agents)}
    
    for case in test_cases:
        for i in range(num_agents):
            pred = models[i].get_value(case.state)
            true = case.true_values[i]
            agent_preds[i].append(pred)
            agent_trues[i].append(true)
    
    def compute_metrics(preds: List[float], trues: List[float]) -> Dict:
        """Compute all metrics for a list of predictions and true values."""
        preds_np = np.array(preds)
        trues_np = np.array(trues)
        
        errors = preds_np - trues_np
        abs_errors = np.abs(errors)
        
        # Non-relative metrics
        mae = float(np.mean(abs_errors))
        std_error = float(np.std(abs_errors))
        max_error = float(np.max(abs_errors))
        mean_bias = float(np.mean(errors))
        std_bias = float(np.std(errors))
        rmse = float(np.sqrt(np.mean(errors ** 2)))
        mean_pred = float(np.mean(preds_np))
        mean_true = float(np.mean(trues_np))
        
        # Relative metrics (only where |true| > EPS)
        valid_mask = np.abs(trues_np) > EPS
        num_valid = int(np.sum(valid_mask))
        
        if num_valid > 0:
            valid_trues = trues_np[valid_mask]
            valid_preds = preds_np[valid_mask]
            valid_errors = valid_preds - valid_trues
            valid_abs_errors = np.abs(valid_errors)
            
            mape = float(np.mean(valid_abs_errors / np.abs(valid_trues)))
            pct_bias = float(np.mean(valid_errors / np.abs(valid_trues)))
        else:
            mape = np.nan
            pct_bias = np.nan
        
        return {
            'mae': mae,
            'std_error': std_error,
            'max_error': max_error,
            'mean_bias': mean_bias,
            'std_bias': std_bias,
            'rmse': rmse,
            'mean_pred': mean_pred,
            'mean_true': mean_true,
            'mape': mape,
            'pct_bias': pct_bias,
            'num_valid_pct_samples': num_valid
        }
    
    results = {}
    
    # Per-agent metrics
    for i in range(num_agents):
        results[f'agent_{i}'] = compute_metrics(agent_preds[i], agent_trues[i])
    
    # Overall metrics (flatten all predictions)
    all_preds = [p for i in range(num_agents) for p in agent_preds[i]]
    all_trues = [t for i in range(num_agents) for t in agent_trues[i]]
    results['overall'] = compute_metrics(all_preds, all_trues)
    
    return results

In [8]:
# =============================================================================
# CSV LOGGING
# =============================================================================

def get_ram_mb() -> float:
    """Get current process RAM usage in MB."""
    return psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)


def save_row(filepath: Path, row_dict: Dict, first_write: bool) -> None:
    """Save a single row to CSV."""
    df = pd.DataFrame([row_dict])
    mode = 'w' if first_write else 'a'
    header = first_write
    df.to_csv(filepath, mode=mode, header=header, index=False)

In [9]:
# =============================================================================
# INIT LOGGING
# =============================================================================

# Create output directory
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(exist_ok=True)

# CSV filename
base_name = f"step1_debug_s{SEED}_pid{os.getpid()}" # may change for convinience
csv_filename = f"{base_name}.csv"
csv_path = output_dir / csv_filename
json_path = output_dir / f"{base_name}.json"

# Save Metadata JSON to identify runs
meta_data = {
    "HEIGHT": HEIGHT,
    "WIDTH": WIDTH,
    "NUM_AGENTS": NUM_AGENTS,
    "NUM_APPLES": NUM_APPLES,
    "PICKER_REWARD": PICKER_REWARD,
    "LR": LR,
    "GAMMA": GAMMA,
    "TRAIN_STEPS": TRAIN_STEPS,
    "EVAL_FREQ": EVAL_FREQ,
    "HIDDEN_DIM": HIDDEN_DIM,
    "NUM_LAYERS": NUM_LAYERS,
    "NUM_TEST_CASES": NUM_TEST_CASES,
    "SEED": SEED,
    "OUTPUT_DIR": OUTPUT_DIR,
    "DEVICE": str(DEVICE)
}

with open(json_path, 'w') as f:
    json.dump(meta_data, f, indent=4)

print(f"Logging to: {csv_path}")
print(f"Metadata saved to: {json_path}")


Logging to: outputs/step1_debug_s42_pid841973.csv
Metadata saved to: outputs/step1_debug_s42_pid841973.json


In [ ]:
# =============================================================================
# INIT ENVIRONMENT, MODELS, TEST CASES
# =============================================================================

# Initialize apple grid (fixed)
apple_grid = init_apple_grid(HEIGHT, WIDTH, NUM_APPLES)
non_apple_positions = get_valid_positions(apple_grid, on_apple=False)
print(f"Apple cells: {int(apple_grid.sum())}, Non-apple cells: {len(non_apple_positions)}")

# Initialize state
state = init_state(HEIGHT, WIDTH, NUM_AGENTS, apple_grid, non_apple_positions)
print(f"\nInitial state:")
print(state)

# Initialize models (one per agent)
models: list[ValueMLP] = []
for i in range(NUM_AGENTS):
    m = ValueMLP(
        self_agent_idx=i,
        height=HEIGHT,
        width=WIDTH,
        hidden_dim=HIDDEN_DIM,
        num_layers=NUM_LAYERS,
        lr=LR,
        gamma=GAMMA,
        device=DEVICE
    )
    models.append(m)

num_params = sum(p.numel() for p in models[0].net.parameters())
print(f"\nModels: {NUM_AGENTS} x {num_params:,} parameters")

# Generate test cases
test_cases = generate_test_cases(
    n=NUM_TEST_CASES,
    height=HEIGHT,
    width=WIDTH,
    num_agents=NUM_AGENTS,
    apples=apple_grid,
    true_value_fn=true_value_fn_step1,
    valid_positions=non_apple_positions  # add this parameter
)
print(f"Test cases: {len(test_cases)}")

Apple cells: 40, Non-apple cells: 41

Initial state:
State (H=9, L=9, actor=0)
Agent positions:
  Agent 0: (np.int64(8), np.int64(7)) <-- ACTOR
  Agent 1: (np.int64(6), np.int64(5))
  Agent 2: (np.int64(3), np.int64(0))
  Agent 3: (np.int64(0), np.int64(1))
Total apples: 40

Models: 4 x 30,209 parameters
Test cases: 200


In [11]:
# =============================================================================
# TRAINING LOOP
# =============================================================================

first_write = True

def log_step(step: int, update_info: List[Dict]):
    global first_write
    eval_results = evaluate(models, test_cases)
    
    # Average update info across buffer
    avg_loss = np.mean([u['loss'] for u in update_info]) if update_info else 0.0
    avg_v_curr = np.mean([u['v_curr'] for u in update_info]) if update_info else 0.0
    avg_td_target = np.mean([u['td_target'] for u in update_info]) if update_info else 0.0
    
    row = {
        'step': step,
        'loss_avg': avg_loss,
        'v_curr_avg': avg_v_curr,
        'td_target_avg': avg_td_target,
        'lr': LR,
        'ram_mb': get_ram_mb()
    }
    for key, metrics in eval_results.items():
        for metric_name, value in metrics.items():
            row[f'{key}_{metric_name}'] = value
    
    save_row(csv_path, row, first_write)
    first_write = False

# Step 0
log_step(0, [])

update_buffer = []

for step in range(1, TRAIN_STEPS + 1):
    rewards = get_reward(state)
    
    next_state = state.copy()
    teleport_and_advance(next_state, non_apple_positions, NUM_AGENTS)
    
    for i in range(NUM_AGENTS):
        info = models[i].update(state, rewards[i], next_state)
        update_buffer.append(info)
    
    state = next_state
    
    if step % EVAL_FREQ == 0:
        log_step(step, update_buffer)
        update_buffer = []

In [12]:
# =============================================================================
# FINAL ANALYSIS
# =============================================================================

print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)

final_eval = evaluate(models, test_cases)

print(f"\n{'Agent':<10} {'MAE':<12} {'Max Err':<12} {'Bias':<12} {'Mean Pred':<12}")
print("-" * 58)
for i in range(NUM_AGENTS):
    r = final_eval[f'agent_{i}']
    print(f"{i:<10} {r['mae']:<12.6f} {r['max_error']:<12.6f} {r['mean_bias']:<+12.6f} {r['mean_pred']:<12.6f}")
print("-" * 58)
r = final_eval['overall']
print(f"{'Overall':<10} {r['mae']:<12.6f} {r['max_error']:<12.6f} {r['mean_bias']:<+12.6f} {r['mean_pred']:<12.6f}")

print(f"\nExpected: All values should be ~0 (true value = 0)")
print(f"RAM: {get_ram_mb():.1f} MB")


FINAL RESULTS

Agent      MAE          Max Err      Bias         Mean Pred   
----------------------------------------------------------
0          0.000333     0.001160     +0.000275    0.000275    
1          0.004563     0.006925     +0.004563    0.004563    
2          0.002710     0.006892     +0.002458    0.002458    
3          0.002897     0.004591     +0.002897    0.002897    
----------------------------------------------------------
Overall    0.002626     0.006925     +0.002548    0.002548    

Expected: All values should be ~0 (true value = 0)
RAM: 677.1 MB


In [13]:
# =============================================================================
# SAMPLE PREDICTIONS
# =============================================================================

print("all apple locations:")
print(apple_grid)

print("\n" + "="*60)
print("SAMPLE PREDICTIONS (First 5 test cases)")
print("="*60)

for idx, case in enumerate(test_cases[:5]):
    print(f"\n--- Test Case {idx} ---")
    print(case.state)
    print("\nPredictions vs True:")
    for i in range(NUM_AGENTS):
        pred = models[i].get_value(case.state)
        true = case.true_values[i]
        print(f"  Agent {i}: pred={pred:+.6f}, true={true:.1f}, error={pred-true:+.6f}")

all apple locations:
[[1. 0. 0. 0. 1. 1. 0. 1. 0.]
 [1. 1. 0. 1. 1. 0. 0. 1. 0.]
 [1. 1. 0. 0. 1. 0. 0. 1. 0.]
 [0. 1. 0. 1. 1. 0. 1. 1. 1.]
 [0. 0. 0. 1. 1. 0. 1. 0. 1.]
 [1. 0. 1. 0. 1. 1. 0. 0. 1.]
 [1. 1. 1. 0. 0. 0. 0. 1. 1.]
 [0. 1. 0. 1. 1. 1. 0. 1. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 1.]]

SAMPLE PREDICTIONS (First 5 test cases)

--- Test Case 0 ---
State (H=9, L=9, actor=3)
Agent positions:
  Agent 0: (np.int64(1), np.int64(5))
  Agent 1: (np.int64(1), np.int64(8))
  Agent 2: (np.int64(5), np.int64(3))
  Agent 3: (np.int64(0), np.int64(1)) <-- ACTOR
Total apples: 40

Predictions vs True:
  Agent 0: pred=-0.000009, true=0.0, error=-0.000009
  Agent 1: pred=+0.004813, true=0.0, error=+0.004813
  Agent 2: pred=+0.001424, true=0.0, error=+0.001424
  Agent 3: pred=+0.003050, true=0.0, error=+0.003050

--- Test Case 1 ---
State (H=9, L=9, actor=3)
Agent positions:
  Agent 0: (np.int64(1), np.int64(6))
  Agent 1: (np.int64(5), np.int64(3))
  Agent 2: (np.int64(2), np.int64(3))
  Agent 3: (